**Travel Planner**

*Multi-AI Agent System using CrewAI*

Agents Involved
1. Travel Preference Analyst
2. Destination Research Specialist
3. Transportation Planning Specialist
4. Accommodation and Budget Specialist
5. Itinerary Generation Specialist
6. Safety and Risk Assessment Specialist
7. Travel Report Generation Specialist


In [42]:
!pip install -q crewai crewai-tools

import os
import warnings

from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, ScrapeWebsiteTool

os.environ["GROQ_API_KEY"] = "gsk...GgVb"
os.environ["SERPER_API_KEY"] = "183...f0b5"

llm = LLM(
    model="groq/llama-3.3-70b-versatile",
    api_key= os.environ["GROQ_API_KEY"],
    max_tokens= 900
)
llm_tools = LLM(
    model="groq/openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    max_tokens= 450
)

search_tool = SerperDevTool()
scrape_tool = ScrapeWebsiteTool()

warnings.filterwarnings("ignore")

print("✅ Setup complete!")

✅ Setup complete!


In [43]:
import crewai
import crewai_tools

print("CrewAI version:", crewai.__version__)
print("CrewAI Tools imported successfully!")

CrewAI version: 1.15.2
CrewAI Tools imported successfully!


In [44]:
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

**Preference Analyzer Agent**

*Understands user preferences, budget, interests, travel style, number of travelers, and constraints.*

In [45]:
preference_agent = Agent(
    role="Travel Preference Analyst",

    goal="""
    Understand the traveler's preferences, priorities, budget constraints,
    and travel style in order to create a personalized travel profile that
    guides all subsequent planning decisions.
    """,

    backstory="""
    You interpret traveler requirements and identify what matters most —
    adventure, culture, food, or budget — turning raw input into a
    structured profile the rest of the team can build on.
    """,

    llm=llm,
    verbose=False,
    memory=False,
    max_iter=1,
    allow_delegation=False
)

**Destination Research Agent**

*Researches attractions, hidden gems, local events, weather, and destination-specific information.*

In [46]:
destination_agent = Agent(
    role="Destination Research Specialist",
    goal="""
    Gather comprehensive and up-to-date information about the destination,
    including attractions, local experiences, events, and activities that
    align with the traveler's preferences and interests.
    """,
    backstory="""
    You're an expert travel researcher who finds both famous landmarks and
    hidden gems, always tailored to the traveler's interests.
    """,

    llm=llm_tools,
    verbose=False,
    memory=False,
    max_iter=2,
    allow_delegation=False
)

**Transportation Planning Specialist**

*Designs efficient and cost-effective transportation plans for seamless travel between locations.*

In [47]:
transport_agent = Agent(
    role="Transportation Planning Specialist",

    goal="""
    Design efficient and cost-effective transportation plans that minimize
    travel time while aligning with the traveler's budget, preferences,
    and itinerary requirements.
    """,

    backstory="""
    You optimize how travelers move between locations, balancing cost,
    duration, and convenience across the whole trip.
    """,

    llm=llm,
    verbose=False,
    memory=False,
    max_iter=2,
    allow_delegation=False
)

**Accommodation and Budget Specialist**

*Recommends accommodations and optimizes expenses to maximize value within the traveler's budget.*

In [48]:
accommodation_budget_agent = Agent(
    role="Accommodation and Budget Specialist",

    goal="""
    Recommend accommodations and optimize trip expenses to maximize value
    while ensuring that the travel plan remains within the traveler's
    financial constraints.
    """,

    backstory="""
    You balance comfort, cost, and convenience when recommending
    accommodations and allocating trip budgets.
    """,

    llm=llm,
    verbose=False,
    memory=False,
    max_iter=2,
    allow_delegation=False
)

**Itinerary Generation Specialist**

*Creates a balanced and personalized day-wise itinerary by integrating recommendations from all planning agents.*

In [49]:
itinerary_agent = Agent(
    role="Itinerary Generation Specialist",

    goal="""
    Create a coherent, engaging, and efficient travel itinerary that balances
    exploration, relaxation, and logistical feasibility while reflecting the
    traveler's preferences and priorities.
    """,

    backstory="""
    You turn other agents' recommendations into a seamless day-by-day plan,
    minimizing unnecessary travel and complexity.
    """,

    llm=llm,
    verbose=False,
    memory=False,
    max_iter=1,
    allow_delegation=False
)

**Safety and Risk Assessment Specialist**

*Evaluates travel advisories, weather conditions, and potential risks to improve trip reliability and safety.*

In [50]:
safety_agent = Agent(
    role="Safety and Risk Assessment Specialist",

    goal="""
    Identify potential risks, travel advisories, weather concerns, and
    logistical challenges in order to improve the safety and reliability
    of the travel plan.
    """,

    backstory="""
    You flag destination-specific risks and give preventive tips so
    travelers stay informed and prepared.
    """,

    llm=llm_tools,
    verbose=False,
    memory=False,
    max_iter=2,
    allow_delegation=False
)

**Travel Report Generation Specialist**

*Consolidates outputs from all agents into a clear, organized, and actionable travel report.*

In [51]:
summary_agent = Agent(
    role="Travel Report Generation Specialist",

    goal="""
    Consolidate outputs from all planning agents into a clear, organized,
    and actionable travel report that provides travelers with a complete
    overview of their journey.
    """,

    backstory="""
     You're the final integration layer, combining every agent's output
    into one easy-to-follow travel dossier.
    """,

    llm=llm,
    verbose=False,
    memory=False,
    max_iter=1,
    allow_delegation=False
)

**Task 1 — Analyze Traveler Preferences**

In [52]:
preference_task = Task(
    description="""
    Analyze the user's travel preferences including destination : {destination}, budget: {budget},
    duration: {duration}, number of travelers: {travelers}, travel style, interests: {interests}, and constraints.

    Generate a structured traveler profile that can guide subsequent
    planning decisions.
    """,

    expected_output="""
    A structured traveler profile including:
    - Traveler type
    - Budget category
    - Interests
    - Preferred travel pace
    - Constraints and priorities
    """,

    agent=preference_agent
)

**Task 2 — Research Destination**

In [53]:
destination_task = Task(
    description="""
    Research the destination that is, {destination} and identify attractions, experiences,
    local events, hidden gems, and destination-specific insights
    relevant to the traveler's preferences that are {interests}.
    """,

    expected_output="""
    A destination research report including:
    - Major attractions
    - Hidden gems
    - Local events
    - Recommended activities
    - Destination insights
    """,

    agent=destination_agent
)

**Task 3 — Plan Transportation**

In [54]:
transport_task = Task(
    description="""
    The destination is {destination}, trip duration is {duration}, for
    {travelers} traveler(s), with interests in: {interests}.

    Design transportation plans including flights, local transportation,
    airport transfers, and travel routes while optimizing cost and travel time.
    """,

    expected_output="""
    A transportation plan including:
    - Recommended flights
    - Local transportation methods
    - Airport transfers
    - Estimated transportation costs
    """,

    agent=transport_agent
)

**Task 4 — Recommend Accommodation and Optimize Budget**

In [55]:
accommodation_budget_task = Task(
    description="""
     The destination is {destination}, trip duration is {duration}, for
    {travelers} traveler(s), with interests in: {interests}.

    Recommend suitable accommodations and optimize expenses to ensure
    that the travel plan remains within budget constraints.
    """,

    expected_output="""
    A report including:
    - Accommodation recommendations
    - Cost breakdown
    - Budget allocation
    - Cost optimization suggestions
    """,

    agent=accommodation_budget_agent
)

**Task 5 — Generate Itinerary**

In [56]:
itinerary_task = Task(
    description="""
    The destination is {destination}, trip duration is {duration}, for
    {travelers} traveler(s), with interests in: {interests}.
    Create a day-wise travel itinerary by integrating destination
    recommendations, transportation logistics, accommodations,
    and traveler preferences.
    """,

    expected_output="""
    A detailed day-by-day itinerary including:
    - Activities
    - Travel times
    - Suggested timings
    - Daily schedules
    """,

    agent=itinerary_agent
)

**Task 6 — Perform Safety Assessment**

In [57]:
safety_task = Task(
    description="""
   The destination is {destination} and the trip duration is {duration}, preferences are {interests}.
    Evaluate safety risks, weather conditions, travel advisories,
    and logistical challenges that may impact the trip.
    """,

    expected_output="""
    A safety report including:
    - Weather concerns
    - Travel advisories
    - Risk mitigation recommendations
    - Safety tips
    """,

    agent=safety_agent
)

**Task 7 — Generate Final Travel Report**

In [58]:
summary_task = Task(
    description="""
    Consolidate all outputs into a final travel report that provides
    travelers with a complete overview of their journey.
    """,

    expected_output="""
    A final travel dossier containing:
    - Traveler profile
    - Destination recommendations
    - Transportation plan
    - Accommodation details
    - Budget summary
    - Itinerary
    - Safety recommendations
    """,

    agent=summary_agent
)

In [59]:
crew = Crew(
    agents=[
        preference_agent,
        destination_agent,
        transport_agent,
        accommodation_budget_agent,
        itinerary_agent,
        summary_agent
    ],

    tasks=[
        preference_task,
        destination_task,
        transport_task,
        accommodation_budget_task,
        itinerary_task,
        safety_task,
        summary_task
    ],

    process=Process.sequential,
    verbose=True,
    max_rpm=4
)

In [60]:
result = await crew.kickoff_async(
    inputs={
        "destination": "Switzerland",
        "budget": "300000 INR",
        "duration": "4 nights, 5 days",
        "travelers": 2,
        "interests": "chocolate, adventure, photography"
    }
)

print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 2fbdc866-3373-466b-bb90-dbf06beb187c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      Analyze the user's travel preferences including destination : Switzerland, budget: 300000 INR,             │
│      duration: 4 nights, 5 days, number of travelers: 2, travel style, interests: chocolate, adventure,         │
│  photography, and constraints.                                                                                  │
│                                                                                                                 │
│      Generate a structured traveler profile that can guide subsequent                                           │
│      planning decisions.                                                                                        │
│                                                                                                                 │
│  ID: bff4e32c-adbb-4702-8b53-2e94c1fa23ab                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Preference Analyst                                                                               │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Analyze the user's travel preferences including destination : Switzerland, budget: 300000 INR,             │
│      duration: 4 nights, 5 days, number of travelers: 2, travel style, interests: chocolate, adventure,         │
│  photography, and constraints.                                                                                  │
│                                                                                                                 │
│      Generate a structured traveler profile that can guide subsequent                                           │
│      planning decisions.                                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Preference Analyst                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Traveler Profile: Switzerland Trip**                                                                         │
│                                                                                                                 │
│  **Traveler Type:** Adventure Seeker and Food Enthusiast                                                        │
│                                                                                                                 │
│  The travelers are a couple looking for a mix of adventure, cultural experiences, and culinary delights, with   │
│  a special interest in chocolate. They are likely to be active, enthusiastic, and eager to explore new          │
│  destinations.                                                                                                  │
│                                                                                                                 │
│  **Budget Category:** Mid-Range to Luxury                                                                       │
│                                                                                                                 │
│  With a budget of 300,000 INR for 4 nights and 5 days, the travelers fall into the mid-range to luxury          │
│  category. This budget allows for comfortable accommodations, some high-end experiences, and flexibility to     │
│  indulge in local cuisine and activities.                                                                       │
│                                                                                                                 │
│  **Interests:**                                                                                                 │
│                                                                                                                 │
│  1. **Chocolate:** The travelers have a sweet tooth and are interested in exploring Switzerland's famous        │
│  chocolate scene, including visiting chocolate factories, taking chocolate-making workshops, or sampling local  │
│  chocolate delicacies.                                                                                          │
│  2. **Adventure:** They enjoy outdoor activities, such as hiking, skiing, or biking, and are likely to want to  │
│  explore Switzerland's stunning mountains, valleys, and lakes.                                                  │
│  3. **Photography:** The travelers are photography enthusiasts, looking to capture the breathtaking             │
│  landscapes, picturesque villages, and vibrant city life in Switzerland.                                        │
│                                                                                                                 │
│  **Preferred Travel Pace:** Moderate to Fast                                                                    │
│                                                                                                                 │
│  Given the short duration of the trip (4 nights, 5 days), the travelers will likely want to make the most of    │
│  their time in Switzerland. They will prefer a moderate to fast travel pace, with a mix of planned activities,  │
│  free time, and flexibility to adjust their itinerary as needed.                                                │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Analyze the user's travel preferences including destination : Switzerland, budget: 300000 INR,             │
│      duration: 4 nights, 5 days, number of travelers: 2, travel style, interests: chocolate, adventure,         │
│  photography, and constraints.                                                                                  │
│                                                                                                                 │
│      Generate a structured traveler profile that can guide subsequent                                           │
│      planning decisions.                                                                                        │
│                                                                                                                 │
│  Agent: Travel Preference Analyst                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Report Generation Specialist                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Travel Report: Switzerland Trip for Adventure Seekers and Food Enthusiasts**                                 │
│                                                                                                                 │
│  **Traveler Profile:**                                                                                          │
│  The travelers are a couple looking for a mix of adventure, cultural experiences, and culinary delights, with   │
│  a special interest in chocolate. They are likely to be active, enthusiastic, and eager to explore new          │
│  destinations.                                                                                                  │
│                                                                                                                 │
│  **Destination Recommendations:**                                                                               │
│  Based on the traveler profile, we recommend visiting the following destinations in Switzerland:                │
│                                                                                                                 │
│  * Zurich: Known for its historic old town, picturesque lake, and mouth-watering chocolate                      │
│  * Interlaken: A popular destination for outdoor activities, such as hiking, paragliding, and boat rides on     │
│  Lake Thun or Lake Brienz                                                                                       │
│  * Geneva: A city famous for its stunning lake, historic old town, and iconic landmarks like the Jet d'Eau      │
│                                                                                                                 │
│  **Transportation Plan:**                                                                                       │
│  To get around Switzerland, we recommend using a combination of public transportation and private transfers.    │
│  The Swiss Travel Pass is a convenient and cost-effective option for traveling by train, bus, and tram.         │
│                                                                                                                 │
│  * Flights: Book flights to Zurich Airport (ZRH) with Swiss International Air Lines or Lufthansa                │
│  * Local Transportation: Use public transportation, such as trains, buses, and trams, to get around             │
│  Switzerland                                                                                                    │
│  * Private Transfers: Book private transfers for airport pickups and drop-offs, as well as for long-distance    │
│  journeys                                                                                                       │
│                                                                                                                 │
│  **Accommodation Details:**                                                                                     │
│  We recommend staying at the following hotels:                                                                  │
│                                                                                                                 │
│  * Hotel du Theatre in Zurich: A 4-star hotel located in the heart of the old town, offering comfortable rooms  │
│  and a rich breakfast buffet                           

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Consolidate all outputs into a final travel report that provides                                           │
│      travelers with a complete overview of their journey.                                                       │
│                                                                                                                 │
│  Agent: Travel Report Generation Specialist                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Travel Report: Switzerland Trip for Adventure Seekers and Food Enthusiasts**

**Traveler Profile:**
The travelers are a couple looking for a mix of adventure, cultural experiences, and culinary delights, with a special interest in chocolate. They are likely to be active, enthusiastic, and eager to explore new destinations.

**Destination Recommendations:**
Based on the traveler profile, we recommend visiting the following destinations in Switzerland:

* Zurich: Known for its historic old town, picturesque lake, and mouth-watering chocolate
* Interlaken: A popular destination for outdoor activities, such as hiking, paragliding, and boat rides on Lake Thun or Lake Brienz
* Geneva: A city famous for its stunning lake, historic old town, and iconic landmarks like the Jet d'Eau

**Transportation Plan:**
To get around Switzerland, we recommend using a combination of public transportation and private transfers. The Swiss Travel Pass is a convenient and cost-effective option for traveling by

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 2fbdc866-3373-466b-bb90-dbf06beb187c                                                                       │
│  Final Output: **Travel Report: Switzerland Trip for Adventure Seekers and Food Enthusiasts**                   │
│                                                                                                                 │
│  **Traveler Profile:**                                                                                          │
│  The travelers are a couple looking for a mix of adventure, cultural experiences, and culinary delights, with   │
│  a special interest in chocolate. They are likely to be active, enthusiastic, and eager to explore new          │
│  destinations.                                                                                                  │
│                                                                                                                 │
│  **Destination Recommendations:**                                                                               │
│  Based on the traveler profile, we recommend visiting the following destinations in Switzerland:                │
│                                                                                                                 │
│  * Zurich: Known for its historic old town, picturesque lake, and mouth-watering chocolate                      │
│  * Interlaken: A popular destination for outdoor activities, such as hiking, paragliding, and boat rides on     │
│  Lake Thun or Lake Brienz                                                                                       │
│  * Geneva: A city famous for its stunning lake, historic old town, and iconic landmarks like the Jet d'Eau      │
│                                                                                                                 │
│  **Transportation Plan:**                                                                                       │
│  To get around Switzerland, we recommend using a combination of public transportation and private transfers.    │
│  The Swiss Travel Pass is a convenient and cost-effective option for traveling by train, bus, and tram.         │
│                                                                                                                 │
│  * Flights: Book flights to Zurich Airport (ZRH) with Swiss International Air Lines or Lufthansa                │
│  * Local Transportation: Use public transportation, such as trains, buses, and trams, to get around             │
│  Switzerland                                                                                                    │
│  * Private Transfers: Book private transfers for airport pickups and drop-offs, as well as for long-distance    │
│  journeys                                                                                                       │
│                                                                                                                 │
│  **Accommodation Details:**                                                                                     │
│  We recommend staying at the following hotels:                                                                  │
│                                                                                                                 │
│  * Hotel du Theatre in Zurich: A 4-star hotel located in the heart of the old town, offering comfortable rooms  │
│  and a rich breakfast buffet                          